# v32 unified one-run — advocate generation + verifier rerank + risk report

Một notebook duy nhất thay cho 4 notebook v32. Chạy theo thứ tự cell từ trên xuống. Cell advocate generation cần GPU; các cell còn lại CPU-only. Notebook luôn giữ baseline locked `v30.1 macro-F1 = 0.5934206145879246` và chỉ select v32 nếu saved preds thật verify được.


In [ ]:
# ==== v32 CORE (shared) ====\n# ============================================================
# v32 CORE - pool-oracle audit + targeted advocate generation + verifier rerank
# LOCKED BASELINE = v30.1 macro-F1 = 0.5934206145879246 (diffs vs v27 = {71,109,125})
# Decision from v31_a data (measured, not assumed):
#   - ALL 18 gold-Yes pred-No errors have ZERO Yes candidates in pool
#   - pool vote signals: precision 0.35-0.64, all below 0.7 gate
#   => selector-only rescue is DEAD for Yes; v32 = generation refresh + verifier rerank
# ============================================================
import json, re, glob, os, random
from collections import Counter, defaultdict

LABELS = ['A','B','C','D','Yes','No','Unknown']
YNU = ['Yes','No','Unknown']
V27_MACRO  = 0.5833102471757934
V301_MACRO = 0.5934206145879246
V301_DIFFS = {71, 109, 125}

def find_file(fname, extra=()):
    for h in extra:
        if os.path.exists(h): return h
    for root in ('/kaggle/working','/kaggle/input'):
        g = glob.glob(f'{root}/**/{fname}', recursive=True)
        if g: return sorted(g)[0]
    raise FileNotFoundError(f'FATAL: {fname} not found - STOP')

def metrics(d):
    tp, fp, fn = Counter(), Counter(), Counter()
    for r in d:
        g, p = r['gold'], r['pred']
        if g == p: tp[g] += 1
        else: fp[p] += 1; fn[g] += 1
    per = {}
    for l in LABELS:
        pr = tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0.0
        rc = tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        per[l] = dict(precision=pr, recall=rc, f1=f1, support=tp[l]+fn[l], pred_count=tp[l]+fp[l])
    tot = sum(v['support'] for v in per.values())
    return dict(n=len(d), acc=sum(r['gold']==r['pred'] for r in d)/len(d),
        macro_f1=sum(v['f1'] for v in per.values())/len(LABELS),
        weighted_f1=sum(v['f1']*v['support'] for v in per.values())/tot,
        mc_macro=sum(per[l]['f1'] for l in 'ABCD')/4,
        ynu_macro=sum(per[l]['f1'] for l in YNU)/3, per_label=per)

def show(m, name):
    print(f"--- {name} ---")
    print(f"acc={m['acc']:.4f} macro-F1={m['macro_f1']:.6f} weighted-F1={m['weighted_f1']:.4f} "
          f"MC_macro={m['mc_macro']:.4f} YNU_macro={m['ynu_macro']:.4f}")
    for l in LABELS:
        v = m['per_label'][l]
        flag = ' COLLAPSE' if v['pred_count']==0 and v['support']>0 else (' LOW_F1' if v['f1']<0.35 else '')
        print(f"  {l:8s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1']:.3f} "
              f"gold={v['support']} pred={v['pred_count']}{flag}")

def load_baseline_v301():
    v27 = json.load(open(find_file('v27_standard_preds.json',
        ['/kaggle/input/datasets/yixuanisthebest/v2333333/v27_standard_preds.json'])))
    assert abs(metrics(v27)['macro_f1'] - V27_MACRO) < 1e-9, 'v27 mismatch, STOP'
    base = json.load(open(find_file('v30_1_full_preds.json')))
    mb = metrics(base)
    diffs = sorted(a['idx'] for a, b in zip(v27, base) if a['pred'] != b['pred'])
    assert abs(mb['macro_f1'] - V301_MACRO) < 1e-9 and set(diffs) == V301_DIFFS, \
        f'v30.1 corrupted (macro={mb["macro_f1"]}, diffs={diffs}), STOP'
    print(f'baseline v30.1 VERIFIED: macro={mb["macro_f1"]:.10f} diffs={diffs}')
    return v27, base, mb

def save_and_verify(preds, summary, tag, v27, expected_diffs=None):
    pp = f'/kaggle/working/{tag}_preds.json'; sp = f'/kaggle/working/{tag}_summary.json'
    json.dump(preds, open(pp,'w'), ensure_ascii=False, indent=1)
    json.dump(summary, open(sp,'w'), indent=1, default=str)
    re_p = json.load(open(pp)); re_m = metrics(re_p)
    assert abs(re_m['macro_f1'] - summary['metrics']['macro_f1']) < 1e-9, f'{tag} SAVE BUG, STOP'
    diffs = sorted(a['idx'] for a, b in zip(v27, re_p) if a['pred'] != b['pred'])
    if expected_diffs is not None:
        assert set(diffs) == set(expected_diffs), f'{tag} saved diffs {diffs} != expected, STOP'
    print(f'SAVED+VERIFIED {tag}_preds.json / {tag}_summary.json (diffs vs v27: {diffs})')

# ---------- target set: where generation refresh applies ----------
def target_cases(base):
    """YNU cases where pred in {No, UNPARSEABLE}, excluding v30.1 banked flips.
    Never touch Unknown preds (Unknown-F1=0.821) and never touch pred=Yes in v32
    (pool-majority signals measured at precision 0.35 - too weak)."""
    return [r for r in base if (not r.get('is_mc'))
            and r['pred'] in ('No','UNPARSEABLE')
            and r['idx'] not in V301_DIFFS]

# ---------- Horn-closure verifier over premises-FOL (reused from v30 core) ----------
def norm_fol(s):
    s = str(s)
    for a,b in [('∀','ForAll '),('∃','Exists '),('→','->'),('⟶','->'),('∧','&'),('∨','|'),('¬','~'),('↔','<->')]:
        s = s.replace(a,b)
    for _ in range(6):
        m = re.search(r'Implies\s*\(([^()]*(?:\([^()]*\)[^()]*)*?),\s*([^()]*(?:\([^()]*\)[^()]*)*?)\)', s)
        if not m: break
        s = s[:m.start()] + f'({m.group(1)} -> {m.group(2)})' + s[m.end():]
    return s

ATOM_RE = re.compile(r'(~?)\s*([A-Z][A-Za-z0-9_]*)\s*\(([^()]*)\)')
RESERVED = {'ForAll','Exists','Implies','And','Or','Not','If','Then'}

def atoms_of(s):
    out = []
    for neg, name, args in ATOM_RE.findall(s):
        if name in RESERVED: continue
        out.append((name, neg=='~'))
    return out

def parse_premise_fol(s):
    s = norm_fol(s)
    quant = 'U' if re.search(r'\bForAll\b', s) else ('E' if re.search(r'\bExists\b', s) else 'F')
    items = []
    if '->' in s:
        ant, cons = s.split('->', 1)
        A, C = atoms_of(ant), atoms_of(cons)
        if A and C:
            ant_pos = frozenset(a for a,n in A if not n)
            for cn, cneg in C:
                items.append(('edge', ant_pos, cn, cneg, quant))
    else:
        for n, neg in atoms_of(s):
            items.append(('fact', n, neg, quant))
    return items

def build_kb(pfs):
    kb = {'edges': [], 'facts': []}
    for pf in (pfs or []):
        try:
            for it in parse_premise_fol(pf):
                (kb['edges'] if it[0]=='edge' else kb['facts']).append(it[1:])
        except Exception: pass
    return kb

def closure(kb, seeds, universal_only):
    derived = set(seeds)
    for n, neg, q in kb['facts']:
        if not neg and (not universal_only or q=='U'): derived.add(n)
    changed = True
    while changed:
        changed = False
        for ant, cons, cneg, q in kb['edges']:
            if cneg or (universal_only and q!='U'): continue
            if ant and ant <= derived and cons not in derived:
                derived.add(cons); changed = True
    return derived

def _stem(w): return w[:-1] if w.endswith('s') and len(w)>3 else w
def split_camel(name):
    return set(_stem(w.lower()) for w in re.findall(r'[A-Z][a-z0-9]*|[a-z]+', name) if len(w)>2)
STOP = set('do does are is the a an all every according premises following statement true based above for that have has will can must of in to with on it exists forall there one least'.split())
def stmt_words(text):
    text = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', str(text))
    return set(_stem(w) for w in re.findall(r'[a-z]+', text.lower()) if len(w)>2 and w not in STOP)

def kb_preds(kb):
    c = set()
    for ant, cons, cneg, q in kb['edges']: c |= set(ant) | {cons}
    for n, neg, q in kb['facts']: c.add(n)
    return c

def match_pred(text_words, kb, used=()):
    best, score = None, 0.0
    for c in kb_preds(kb) - set(used):
        cw = split_camel(c)
        if not cw: continue
        s = len(cw & text_words)/max(1,len(cw))
        if s > score: best, score = c, s
    return (best, score) if score >= 0.5 else (None, score)

def extract_statement(q):
    m = re.search(r'Statement\s*[:\-]\s*(.+)', q, re.S|re.I)
    return (m.group(1) if m else q).strip()

NEG = r"\b(not|cannot|can't|won't|don't|doesn't|never|no longer|fails? to|without|unable)\b"

def horn_verdict(question, kb):
    """PROVE_YES / OVERCLAIM_NO / ABSTAIN - verifier feature, not hard override."""
    if not kb['edges'] and not kb['facts']: return ('ABSTAIN','empty_kb')
    s = extract_statement(question); sn = norm_fol(s); sl = s.lower()
    m = re.search(r'^if\s+(.*?)\s*(?:,\s*then|,|then)\s+(.*?)[.?]?$', sl, re.S)
    if m:
        p_t, q_t = m.group(1), m.group(2)
        pn, qn = bool(re.search(NEG,p_t)), bool(re.search(NEG,q_t))
        P,_ = match_pred(stmt_words(p_t), kb); Q,_ = match_pred(stmt_words(q_t), kb, used=[P] if P else [])
        if not P or not Q: return ('ABSTAIN','pred_match_fail')
        if pn and qn:   # inverse claim ~P->~Q
            return ('OVERCLAIM_NO',f'inverse: premises give {P}=>{Q}') if Q in closure(kb,{P},True) else ('ABSTAIN','inverse_unproven')
        if pn or qn: return ('ABSTAIN','mixed_negation')
        if Q in closure(kb,{P},True): return ('PROVE_YES',f'chain {P}=>{Q}')
        if P in closure(kb,{Q},True): return ('OVERCLAIM_NO',f'converse: premises give {Q}=>{P}')
        return ('ABSTAIN','no_chain')
    if re.search(r'\b(all|every)\b', sl) or re.search(r'\bForAll\b', sn):
        seeds = set()
        mc = re.search(r'\b(?:all|every)\s+([a-z]+)', sl)
        if mc:
            C,_ = match_pred(stmt_words(mc.group(1)), kb)
            if C: seeds.add(C)
        Q,_ = match_pred(stmt_words(sl) - (stmt_words(mc.group(1)) if mc else set()), kb, used=seeds)
        if not Q: return ('ABSTAIN','pred_match_fail')
        if Q in closure(kb, seeds, True): return ('PROVE_YES',f'universal closure has {Q}')
        if Q in closure(kb, seeds, False): return ('OVERCLAIM_NO',f'{Q} only some/instance support')
        return ('ABSTAIN','no_support')
    if re.search(r'\b(there exists|at least one|some)\b', sl) or re.search(r'\bExists\b', sn):
        Q,_ = match_pred(stmt_words(sl), kb)
        if not Q: return ('ABSTAIN','pred_match_fail')
        return ('PROVE_YES',f'witness {Q}') if Q in closure(kb, set(), False) else ('ABSTAIN','no_witness')
    return ('ABSTAIN','atomic')

def load_val_dataset_aligned(preds):
    p = find_file('Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json')
    d = json.load(open(p))
    recs = d if isinstance(d, list) else (d.get('data') or d.get('samples') or list(d.values())[0])
    norm = lambda q: re.sub(r'\s+',' ',str(q).strip().lower())[:300]
    bym = {}
    for rec in recs:
        for k in ('question','query'):
            if k in rec: bym.setdefault(norm(rec[k]), rec); break
    aligned = {r['idx']: bym.get(norm(r['question'])) for r in preds}
    print(f"dataset aligned {sum(v is not None for v in aligned.values())}/{len(preds)}")
    return aligned

def get_fol(rec):
    if not rec: return []
    for k in rec:
        if 'fol' in k.lower() and 'premise' in k.lower():
            v = rec[k]; return v if isinstance(v, list) else [v]
    for k in rec:
        if 'fol' in k.lower():
            v = rec[k]; return v if isinstance(v, list) else [v]
    return []

def bootstrap_gain(base, new, n_boot=400, seed=0):
    def mac(sub):
        tp,fp,fn=Counter(),Counter(),Counter()
        for r in sub:
            g,p=r['gold'],r['pred']
            if g==p: tp[g]+=1
            else: fp[p]+=1; fn[g]+=1
        f1s=[]
        for l in LABELS:
            if tp[l]+fn[l]==0 and tp[l]+fp[l]==0: continue
            pr=tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0; rc=tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0
            f1s.append(2*pr*rc/(pr+rc) if pr+rc else 0)
        return sum(f1s)/len(f1s) if f1s else 0
    rnd=random.Random(seed); n=len(base); gains=[]
    for _ in range(n_boot):
        ix=[rnd.randrange(n) for _ in range(n)]
        gains.append(mac([new[i] for i in ix]) - mac([base[i] for i in ix]))
    gains.sort()
    res=dict(mean=sum(gains)/len(gains), ci_lo=gains[int(.05*len(gains))],
             ci_hi=gains[int(.95*len(gains))], p_positive=sum(g>0 for g in gains)/len(gains))
    print(f"bootstrap gain: mean={res['mean']:+.4f} 90%CI=[{res['ci_lo']:+.4f},{res['ci_hi']:+.4f}] P(gain>0)={res['p_positive']:.2f}")
    return res

def decide(base_m, new_m, boot, name):
    gain = new_m['macro_f1'] - base_m['macro_f1']
    unk_drop = base_m['per_label']['Unknown']['f1'] - new_m['per_label']['Unknown']['f1']
    no_drop  = base_m['per_label']['No']['f1'] - new_m['per_label']['No']['f1']
    collapse = any(v['pred_count']==0 and v['support']>0 for v in new_m['per_label'].values())
    mc_ok = abs(new_m['mc_macro'] - base_m['mc_macro']) < 1e-12
    ok = gain > 0.01 and boot['p_positive'] >= 0.80 and unk_drop <= 0.0 \
         and no_drop <= 0.05 and not collapse and mc_ok
    status = name if ok else 'ROLLBACK_v30_1'
    print(f'DECISION: {status} (gain={gain:+.4f}, P+={boot["p_positive"]:.2f}, '
          f'unk_drop={unk_drop:+.3f}, no_drop={no_drop:+.3f}, collapse={collapse}, mc_frozen={mc_ok})')
    return status, dict(gain=gain, unk_drop=unk_drop, no_drop=no_drop,
                        collapse=collapse, mc_frozen=mc_ok, selected=status)


## 1. v32_a — Pool-oracle audit
Phân loại lỗi YNU còn lại thành: gold có trong pool, không có gold candidate, hoặc parse/pool fail. Không tạo final prediction.


In [ ]:
# ---- v32_a: pool-oracle audit on remaining wrong YNU cases (analysis only) ----
v27, base, mb = load_baseline_v301()
raw = json.load(open(find_file('v23_val_candidates.json',
    ['/kaggle/input/datasets/yixuanisthebest/v2333333/v23_val_candidates.json'])))
recs = raw if isinstance(raw, list) else (raw.get('data') or list(raw.values())[0])
assert len(recs) == len(base), f'positional align broken: {len(recs)} vs {len(base)} - STOP'
FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)

def cands_of(rec):
    for k in ('candidates','cands','outputs','generations','samples','completions'):
        if k in rec and isinstance(rec[k], list): return rec[k]
    return []
def ans_of(c):
    t = c if isinstance(c, str) else json.dumps(c) if not isinstance(c, dict) else ''
    if isinstance(c, dict):
        for k in ('answer','final_answer','final','pred'):
            if k in c and c[k]:
                v = str(c[k]).strip().title()
                if v in LABELS: return v, 'field'
        for k in ('text','output','explanation','completion','response','generation'):
            if k in c and c[k]: t = str(c[k]); break
    m = FINAL_RE.findall(t)
    return (m[-1].title(), 'parsed') if m else (None, 'PARSE_FAIL')

cases, fam = [], Counter()
for r, rec in zip(base, recs):   # positional alignment (v31 lesson)
    if r.get('is_mc') or r['gold'] == r['pred']: continue
    cl = cands_of(rec)
    answers, srcs = [], []
    for c in cl:
        a, s = ans_of(c); answers.append(a); srcs.append(s)
    parse_fails = srcs.count('PARSE_FAIL')
    vote = Counter(a for a in answers if a)
    if not cl: k = 'D_no_pool'
    elif parse_fails == len(cl): k = 'D_parse_fail_all'
    elif vote.get(r['gold'], 0) > 0: k = 'A_gold_in_pool'
    else: k = 'B_no_gold_candidate'
    fam[k] += 1
    cases.append(dict(idx=r['idx'], gold=r['gold'], pred=r['pred'], family=k,
                      vote=dict(vote), n_cands=len(cl), parse_fails=parse_fails,
                      question=r['question'][:140]))

print('=== wrong-YNU pool-oracle audit ===')
print('families:', dict(fam))
for c in sorted(cases, key=lambda x: (x['family'], x['idx'])):
    print(f"  idx={c['idx']:3d} {c['gold']}->{c['pred']:11s} {c['family']:22s} vote={c['vote']}")

nA, nB = fam.get('A_gold_in_pool',0), fam.get('B_no_gold_candidate',0)
nD = fam.get('D_no_pool',0) + fam.get('D_parse_fail_all',0)
if nD > 5: route = 'FIX_PARSER_FIRST'
elif nB >= nA: route = 'v32_candidate_generation_refresh (advocate prompts) + verifier rerank'
else: route = 'v32_standard_reranker over existing pool'
print(f'\nROUTING DECISION: A={nA} B={nB} D={nD} -> {route}')

json.dump(dict(version='v32_a_pool_oracle_audit', families=dict(fam), routing=route,
    n_wrong=len(cases)), open('/kaggle/working/v32_a_pool_oracle_audit_summary.json','w'), indent=1)
json.dump(cases, open('/kaggle/working/v32_a_pool_oracle_audit_cases.json','w'), indent=1)
print('SAVED v32_a_pool_oracle_audit_summary.json / v32_a_pool_oracle_audit_cases.json')


## 2. v32_b — Advocate generation GPU
Sinh 3 advocate Yes/No/Unknown cho target cases. Không retrain. Output chỉ là raw material, không phải prediction final.


In [ ]:
# ---- v32_b: targeted advocate generation (GPU, ~50 cases x 3 prompts) ----
# Same Qwen3-8B + v23 LoRA. NO retrain. enable_thinking=False (match v14/v23 setup).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

v27, base, mb = load_baseline_v301()
aligned = load_val_dataset_aligned(base)
targets = target_cases(base)
print(f'target cases (pred No/UNPARSEABLE, YNU, excl banked): {len(targets)}')

BASE_MODEL = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'
ADAPTER    = '/kaggle/input/datasets/yixuanisthebest/v2333333'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type='nf4')
n_gpu = torch.cuda.device_count()
print('GPUs visible:', n_gpu)
kw = dict(quantization_config=bnb, torch_dtype=torch.float16)
if n_gpu >= 2:
    kw.update(device_map='auto', max_memory={0:'13GiB', 1:'13GiB'})
else:
    kw.update(device_map='auto')   # never hard-code GPU 1 (v24.1 lesson)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
model = PeftModel.from_pretrained(model, ADAPTER)
tok = AutoTokenizer.from_pretrained(ADAPTER)
model.eval()

def premises_text(rec):
    if not rec: return None
    for k in rec:
        if 'premise' in k.lower() and 'fol' not in k.lower():
            v = rec[k]
            return '\n'.join(f'{i+1}. {p}' for i, p in enumerate(v)) if isinstance(v, list) else str(v)
    return None

ADV_PROMPT = (
 'Premises:\n{prem}\n\nQuestion: {q}\n\n'
 'Task: Construct the strongest possible argument that the answer is "{tgt}". '
 'Use ONLY the numbered premises. Cite premise numbers for every inference step. '
 'Watch for converse/inverse errors and some-vs-all (quantifier) errors.\n'
 'Format exactly:\nArgument: <numbered steps citing premises>\n'
 'Supporting Premises: [i, j]\n'
 'Self-Check: <do the cited premises actually entail "{tgt}"? any fallacy?>\n'
 'Verdict: VALID or INVALID')

@torch.no_grad()
def gen(prompt, max_new=400):
    msgs = [{'role':'system','content':'You are a rigorous logician. Be strict: if the premises do not entail the target answer, your Verdict must be INVALID.'},
            {'role':'user','content':prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)
    ids = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

advocates, skipped = {}, []
for t_i, r in enumerate(targets):
    prem = premises_text(aligned.get(r['idx']))
    if not prem:
        skipped.append(r['idx']); continue
    entry = {}
    for tgt in ('Yes','No','Unknown'):
        txt = gen(ADV_PROMPT.format(prem=prem, q=r['question'], tgt=tgt))
        m = re.search(r'Verdict\s*[:\-]?\s*(VALID|INVALID)', txt, re.I)
        cm = re.search(r'Supporting Premises\s*[:\-]?\s*\[([^\]]*)\]', txt)
        entry[tgt] = dict(text=txt, verdict=(m.group(1).upper() if m else 'UNPARSED'),
                          cites=[int(x) for x in re.findall(r'\d+', cm.group(1))] if cm else [])
    advocates[r['idx']] = entry
    if (t_i+1) % 10 == 0: print(f'{t_i+1}/{len(targets)} done')

print(f'generated advocates for {len(advocates)} cases; skipped (no premises): {skipped}')
json.dump(advocates, open('/kaggle/working/v32_b_advocate_candidates.json','w'),
          ensure_ascii=False, indent=1)
print('SAVED v32_b_advocate_candidates.json  (raw material only - NOT predictions)')


## 3. v32_standard — Advocate verifier rerank
CPU-only. Dùng Horn/verdict/citation gates khai báo trước. Không đụng MC, Unknown, pred=Yes, hoặc banked flips v30.1 `{71,109,125}`.


In [ ]:
# ---- v32_standard: verifier-rerank over advocates (no model load) ----
v27, base, mb = load_baseline_v301()
show(mb, 'BASE v30.1 (locked)')
aligned = load_val_dataset_aligned(base)
adv = json.load(open(find_file('v32_b_advocate_candidates.json')))
adv = {int(k): v for k, v in adv.items()}
print(f'advocate cases: {len(adv)}')

def n_premises(rec):
    if not rec: return 0
    for k in rec:
        if 'premise' in k.lower() and 'fol' not in k.lower() and isinstance(rec[k], list):
            return len(rec[k])
    return 0

out, flips, log = [], [], []
for r in base:
    nr = dict(r)
    a = adv.get(r['idx'])
    if a and (not r.get('is_mc')) and r['pred'] in ('No','UNPARSEABLE') and r['idx'] not in V301_DIFFS:
        rec = aligned.get(r['idx'])
        kb = build_kb(get_fol(rec))
        hv, hwhy = horn_verdict(r['question'], kb)
        npr = n_premises(rec)
        def ok_cites(e): return bool(e['cites']) and (npr == 0 or all(0 < c <= npr for c in e['cites']))
        y, n_, u = a.get('Yes',{}), a.get('No',{}), a.get('Unknown',{})
        new_pred, why = None, None
        # S1: Horn proof (highest precision) - verifier evidence, applied to targeted subset only
        if hv == 'PROVE_YES':
            new_pred, why = 'Yes', f'S1_horn:{hwhy}'
        # S2: advocate asymmetry - Yes advocate VALID+cited while No advocate self-judged INVALID
        elif y.get('verdict')=='VALID' and ok_cites(y) and n_.get('verdict')=='INVALID':
            new_pred, why = 'Yes', 'S2_advocate: Yes VALID+cited, No INVALID'
        # S3: UNPARSEABLE only - take unique VALID advocate
        elif r['pred'] == 'UNPARSEABLE':
            valids = [k for k in ('Yes','No','Unknown') if a.get(k,{}).get('verdict')=='VALID' and ok_cites(a.get(k,{}))]
            if len(valids) == 1: new_pred, why = valids[0], 'S3_unique_valid_advocate'
        if new_pred and new_pred != r['pred']:
            nr['pred'] = new_pred; nr['repair'] = f'v32:{why}'
            flips.append((r['idx'], r['pred'], new_pred, r['gold'], why))
        log.append(dict(idx=r['idx'], horn=hv, yes_v=y.get('verdict'), no_v=n_.get('verdict'),
                        unk_v=u.get('verdict'), flipped=bool(new_pred and new_pred != r['pred'])))
    out.append(nr)

print(f'FLIPS={len(flips)}, correct={sum(1 for f in flips if f[2]==f[3])}')
for f in flips: print('  idx=%3d %s->%s gold=%s | %s' % f)
nm = metrics(out); show(nm, 'v32_standard')
boot = bootstrap_gain(base, out)
status, gates = decide(mb, nm, boot, 'v32_standard')
sel, sel_m = (out, nm) if status=='v32_standard' else (base, mb)
expected = sorted(V301_DIFFS | {f[0] for f in flips}) if status=='v32_standard' else sorted(V301_DIFFS)
summary = dict(version='v32_standard_advocate_verifier_rerank', selection_status=status,
    metrics={k: sel_m[k] for k in ('n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro')},
    per_label=sel_m['per_label'], n_flips=len(flips), flips=[list(f) for f in flips],
    rules='S1 horn-proof, S2 advocate-asymmetry, S3 unique-valid (a priori, no grid)',
    decision_log=log, bootstrap=boot, gates=gates)
save_and_verify(sel, summary, 'v32_standard', v27, expected_diffs=expected)


## 4. v32_full — Select or rollback
Select v32_standard chỉ nếu summary và actual saved preds đều pass verification. Nếu không, rollback v30.1.


In [ ]:
# ---- v32_full: select v32_standard only if gates passed AND saved artifact verifies ----
import glob as _g
v27, base, mb = load_baseline_v301()
cand = (_g.glob('/kaggle/working/v32_standard_summary.json')
        or _g.glob('/kaggle/input/**/v32_standard_summary.json', recursive=True))
chosen, src = base, 'ROLLBACK_v30_1 (no v32_standard summary)'
if cand:
    summ = json.load(open(cand[0]))
    if summ.get('selection_status') == 'v32_standard':
        pr = json.load(open(cand[0].replace('summary','preds')))
        pm = metrics(pr)
        if abs(pm['macro_f1'] - summ['metrics']['macro_f1']) < 1e-9 and pm['macro_f1'] > V301_MACRO \
           and abs(pm['mc_macro'] - mb['mc_macro']) < 1e-12:
            chosen, src = pr, 'v32_standard artifact-verified'
        else:
            src = f"ROLLBACK_v30_1 (artifact verification FAILED: saved={pm['macro_f1']:.6f} summary={summ['metrics']['macro_f1']:.6f})"
    else:
        src = f"ROLLBACK_v30_1 (selection_status={summ.get('selection_status')})"
print('SOURCE =', src)
fm = metrics(chosen); show(fm, 'v32_full FINAL')
summary = dict(version='v32_full', source=src,
    metrics={k: fm[k] for k in ('n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro')},
    per_label=fm['per_label'])
save_and_verify(chosen, summary, 'v32_full', v27)


## 5. Final metrics + risk report
In thông số cuối cùng kèm overfit risk, underfit risk, label collapse, artifact risk. Lưu `v32_risk_report.json`.


In [ ]:
# ---- v32 final metrics + overfit/underfit/risk report ----
# This cell is intentionally read-only over saved artifacts.
# It prints the final decision, metrics, and risks, then saves v32_risk_report.json.
import glob, json, math
from collections import Counter

def _find_artifact(name):
    hits = glob.glob(f'/kaggle/working/{name}') or glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

def _load_artifact(name, required=False):
    p = _find_artifact(name)
    if p:
        return json.load(open(p)), p
    if required:
        raise FileNotFoundError(name)
    return None, None

v27, base, mb = load_baseline_v301()
full_s, full_p = _load_artifact('v32_full_summary.json', required=True)
std_s, std_p = _load_artifact('v32_standard_summary.json')
a_s, a_p = _load_artifact('v32_a_pool_oracle_audit_summary.json')
adv_s, adv_p = _load_artifact('v32_b_advocate_candidates.json')

# The full notebook may store final metrics under either `metrics` or `final_metrics` depending on future variants.
fm = full_s.get('metrics') or full_s.get('final_metrics')
if 'per_label' not in fm:
    # recompute from saved preds if summary is compact
    preds, _ = _load_artifact('v32_full_preds.json', required=True)
    fm = metrics(preds)

base_macro = mb['macro_f1']
macro_gain = fm['macro_f1'] - base_macro
acc_gain = fm['acc'] - mb['acc']
mc_delta = fm['mc_macro'] - mb['mc_macro']
ynu_delta = fm['ynu_macro'] - mb['ynu_macro']

# Determine selected/rollback status robustly.
status_text = str(full_s.get('selection_status') or full_s.get('source') or full_s.get('selected_source') or '')
selected_v32 = ('v32_standard' in status_text.lower()) and ('rollback' not in status_text.lower())

# Pull standard gates/flips if present.
std_gates = (std_s or {}).get('gates', {})
std_boot = (std_s or {}).get('bootstrap', {})
std_flips = (std_s or {}).get('flips', [])
n_flips = int((std_s or {}).get('n_flips', len(std_flips) if std_flips else 0))

# Label collapse / weak labels.
per = fm['per_label']
collapse_labels = [k for k,v in per.items() if v.get('support',0)>0 and v.get('pred_count',0)==0]
low_f1_labels = [k for k,v in per.items() if v.get('support',0)>0 and v.get('f1',0)<0.35]
label_collapse = bool(collapse_labels)

# Underfit signals from v32_a.
fam = (a_s or {}).get('families', {})
A_gold = int(fam.get('A_gold_in_pool', 0))
B_no_gold = int(fam.get('B_no_gold_candidate', 0))
D_parse_or_pool = int(fam.get('D_no_pool', 0)) + int(fam.get('D_parse_fail_all', 0))
n_wrong = int((a_s or {}).get('n_wrong', 0))
route = (a_s or {}).get('routing', 'UNKNOWN')

if D_parse_or_pool > 5:
    underfit_risk = 'HIGH_PARSE_OR_POOL_ALIGNMENT_RISK'
elif B_no_gold >= max(A_gold, 1):
    underfit_risk = 'HIGH_CANDIDATE_INSUFFICIENCY'
elif A_gold > B_no_gold and not selected_v32:
    underfit_risk = 'MEDIUM_SELECTOR_GAP'
elif selected_v32 and macro_gain > 0.015:
    underfit_risk = 'REDUCED_BUT_PRESENT'
else:
    underfit_risk = 'MEDIUM'

# Overfit risk: v32 uses VAL-targeted advocate generation + gates; selecting it is inherently validation-tuned.
if selected_v32:
    overfit_risk = 'MEDIUM_HIGH_VALIDATION_TUNED'
    if n_flips > 15:
        overfit_risk = 'HIGH_TOO_MANY_FLIPS'
    elif n_flips <= 10 and std_boot.get('p_positive', 0) >= 0.80 and macro_gain > 0.01:
        overfit_risk = 'MEDIUM_CONTROLLED_BY_GATES'
else:
    overfit_risk = 'LOW_OPERATIONAL_ROLLBACK'

# Artifact risk: final summary must match saved preds if present.
artifact_risk = 'UNKNOWN'
try:
    final_preds, fp = _load_artifact('v32_full_preds.json', required=True)
    recomputed = metrics(final_preds)
    saved_ok = abs(recomputed['macro_f1'] - fm['macro_f1']) < 1e-9 and abs(recomputed['mc_macro'] - fm['mc_macro']) < 1e-12
    artifact_risk = 'LOW_VERIFIED_SAVED_PREDS' if saved_ok else 'HIGH_SUMMARY_PREDS_MISMATCH'
except Exception as e:
    artifact_risk = f'HIGH_COULD_NOT_VERIFY:{type(e).__name__}'

# Overall decision.
if selected_v32:
    final_decision = 'SELECT_V32' if macro_gain > 0.01 and not label_collapse and abs(mc_delta) < 1e-12 else 'ROLLBACK_RECOMMENDED_BY_RISK'
else:
    final_decision = 'ROLLBACK_TO_V30_1'

print('\n' + '='*72)
print('v32 UNIFIED FINAL METRICS + RISK REPORT')
print('='*72)
print(f"Final status       : {status_text}")
print(f"Decision           : {final_decision}")
print(f"Macro-F1           : {fm['macro_f1']:.10f}  (delta vs v30.1 {macro_gain:+.10f})")
print(f"Accuracy           : {fm['acc']:.10f}  (delta {acc_gain:+.10f})")
print(f"Weighted-F1        : {fm['weighted_f1']:.10f}  (delta {fm['weighted_f1']-mb['weighted_f1']:+.10f})")
print(f"MC macro           : {fm['mc_macro']:.10f}  (delta {mc_delta:+.10f})")
print(f"YNU macro          : {fm['ynu_macro']:.10f}  (delta {ynu_delta:+.10f})")
print(f"v32 flips          : {n_flips}")
print(f"Bootstrap p+       : {std_boot.get('p_positive', 'NA')}")
print('\nPer-label F1 / counts:')
for l in LABELS:
    v = per[l]
    flag = ''
    if l in collapse_labels: flag = '  COLLAPSE'
    elif l in low_f1_labels: flag = '  LOW_F1'
    print(f"  {l:8s} F1={v['f1']:.4f} P={v['precision']:.4f} R={v['recall']:.4f} gold={v['support']} pred={v['pred_count']}{flag}")
print('\nRisk assessment:')
print(f"  Overfit risk      : {overfit_risk}")
print(f"  Underfit risk     : {underfit_risk}")
print(f"  Label collapse    : {'YES ' + str(collapse_labels) if label_collapse else 'NO'}")
print(f"  Artifact risk     : {artifact_risk}")
print(f"  Pool audit route  : {route}")
print(f"  Wrong YNU families: A_gold_in_pool={A_gold}, B_no_gold_candidate={B_no_gold}, D_parse_or_pool={D_parse_or_pool}, total={n_wrong}")
print('\nInterpretation:')
if selected_v32:
    print('  v32 selected only if it beats v30.1 and artifact verification passed. Still treat as validation-tuned; confirm on hidden/private split.')
else:
    print('  v32 did not safely beat v30.1; operational output remains v30.1. Remaining gap is candidate-generation / selector underfit, not MC.')
print('='*72)

risk_report = dict(
    version='v32_unified_risk_report',
    final_status=status_text,
    final_decision=final_decision,
    metrics={k: fm[k] for k in ('n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro')},
    delta_vs_v30_1=dict(acc=acc_gain, macro_f1=macro_gain, weighted_f1=fm['weighted_f1']-mb['weighted_f1'], mc_macro=mc_delta, ynu_macro=ynu_delta),
    per_label=per,
    n_flips=n_flips,
    bootstrap=std_boot,
    gates=std_gates,
    overfit_risk=overfit_risk,
    underfit_risk=underfit_risk,
    label_collapse=label_collapse,
    collapse_labels=collapse_labels,
    low_f1_labels=low_f1_labels,
    artifact_risk=artifact_risk,
    pool_audit=dict(families=fam, routing=route, n_wrong=n_wrong),
    recommendation=final_decision,
)
json.dump(risk_report, open('/kaggle/working/v32_risk_report.json','w'), indent=1, ensure_ascii=False, default=str)
print('SAVED v32_risk_report.json')
